# Radar Cidadão — RPA + IA | Demonstração executável
**FIAP — Global Solution 2026.2 • Disciplina AI for Robotic Process Automation**

**Integrantes:** Mario Antônio de Oliveira (567713), Sarah Marangoni Furlan (566764), Vinicius Augusto Silva (566830).

**Objetivo:** automatizar a triagem de relatos *sintéticos* sobre chuvas, enchentes e deslizamentos, indicando orientação geral, necessidade de encaminhamento e casos fora do escopo. **ODS 11**, com relação ao **ODS 13**.

**Como usar:** no Google Colab, selecione **Ambiente de execução → Executar tudo**. CPU é suficiente; não precisa de GPU, token ou Google Drive. As células demonstram entrada, processamento, testes, falhas e evidências. A execução não modifica o GitHub.

> **Limites:** protótipo acadêmico; não prevê desastres, não envia chamados à Defesa Civil e não deve receber dados reais de cidadãos. Não substitui orientações oficiais em emergência.

## 1. Temas da disciplina efetivamente aplicados

| Tema estudado | Aplicação no fluxo | Onde verificar |
|---|---|---|
| Automação de arquivos com Python | Leitura do CSV, geração de CSV e JSON | `radar_rpa/cli.py` |
| RPA com Python e hiperautomação | Orquestração automática, logs e tratamento de falhas | `radar_rpa/cli.py` |
| IA/ML na automação | Classificador TF-IDF + regressão logística | `radar_rpa/core.py` |
| Banco de dados | SQLite para idempotência e histórico por hash | `radar_rpa/cli.py` |
| BPM / gestão de processos | Entrada → validação → classificação → decisão → registro | `docs/FLUXO_BPMN.md` |

As orientações são conservadoras e não constituem previsão de risco.

In [1]:
# 2. Obter o projeto público e instalar as dependências (CPU)
import subprocess, sys, json, csv, tempfile
from pathlib import Path
from IPython.display import display, Markdown

REPO = Path('/content/radar-cidadao')
URL = 'https://github.com/sarahmfurlan-sudo/radar-cidadao.git'
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
PROJETO = REPO / 'rpa-radar-cidadao'
assert (PROJETO / 'pyproject.toml').is_file(), 'Pasta RPA não encontrada no repositório.'
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-e', f'{PROJETO}[test]'], check=True)
COMMIT = subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'], text=True).strip()
print('Projeto:', PROJETO)
print('Commit:', COMMIT)
print('Python:', sys.version.split()[0])
print('Tudo pronto. Execução local: sem GPU e sem API.')

Projeto: /content/radar-cidadao/rpa-radar-cidadao
Commit: 66fa473
Python: 3.13.18
Tudo pronto. Execução local: sem GPU e sem API.


## 3. Entrada de dados sintéticos
O arquivo de demonstração possui **12 linhas** e inclui propositalmente uma entrada inválida. Isso prova que um erro isolado não interrompe a execução inteira.

In [2]:
ENTRADA = PROJETO / 'dados' / 'ocorrencias_demo.csv'
with ENTRADA.open(encoding='utf-8-sig', newline='') as f:
    linhas = list(csv.DictReader(f, delimiter=';'))
print('Ocorrências de demonstração:', len(linhas))
for item in linhas[:5]:
    print(f"{item['id']}: {item['relato']}")
assert len(linhas) >= 1

Ocorrências de demonstração: 12
OC001: A rua da escola esta alagada depois do temporal
OC002: O barranco atras da casa apresenta rachaduras
OC003: Recebi aviso de chuva forte e relampagos
OC004: Estou preso dentro de casa e preciso de socorro
OC005: Como renovar meu documento de identidade?


## 4. Executar o robô do início ao fim
Cada execução do notebook usa uma pasta temporária nova, evitando resultados antigos. O processamento é feito pela **CLI original do projeto**, não por uma simulação reescrita no notebook.

In [3]:
SAIDA = Path(tempfile.mkdtemp(prefix='radar_rpa_colab_'))
DEMO = SAIDA / 'demo'
cmd = [sys.executable, '-m', 'radar_rpa', 'run', '--input', str(ENTRADA), '--output', str(DEMO)]
r = subprocess.run(cmd, cwd=PROJETO, text=True, capture_output=True)
print(r.stdout)
if r.stderr: print('stderr:', r.stderr)
assert r.returncode == 0, 'Falha na execução principal'
RESUMO = json.loads((DEMO / 'resumo.json').read_text(encoding='utf-8'))
print('Arquivos de evidência:', [p.name for p in sorted(DEMO.iterdir())])
assert RESUMO['total_linhas'] == len(linhas)
assert RESUMO['processados'] + RESUMO['invalidos'] + RESUMO['duplicados'] == len(linhas)
print('Fluxo ponta a ponta validado.')

{
  "total_linhas": 12,
  "processados": 11,
  "invalidos": 1,
  "duplicados": 0,
  "por_status": {
    "orientacao_geral": 7,
    "encaminhar_imediatamente": 2,
    "fora_escopo": 2
  },
  "tempo_segundos": 0.327,
  "saida": "/tmp/radar_rpa_colab_gxh35jb_/demo/resultados.csv",
  "erros": []
}

Arquivos de evidência: ['auditoria.sqlite3', 'execucao.log', 'resultados.csv', 'resumo.json']
Fluxo ponta a ponta validado.


## 5. Conferir as saídas e as decisões
Os registros são mascarados parcialmente. **Não use dados pessoais reais**: nomes, endereços e outros identificadores não são eliminados com garantia.

In [4]:
with (DEMO / 'resultados.csv').open(encoding='utf-8-sig', newline='') as f:
    resultados = list(csv.DictReader(f, delimiter=';'))
print('Registros gravados:', len(resultados))
try:
    import pandas as pd
    display(pd.DataFrame(resultados))
except ImportError:
    for item in resultados: print(item)
print('Resumo de status:', RESUMO['por_status'])

Registros gravados: 11


,id,relato_anonimizado,categoria,confianca,status,orientacao
0,OC001,A rua da escola esta alagada depois do temporal,alagamento,0.388,orientacao_geral,Evite atravessar vias alagadas e contato com a...
1,OC002,O barranco atras da casa apresenta rachaduras,deslizamento,0.518,orientacao_geral,"Afaste-se de encostas, barreiras e estruturas ..."
2,OC003,Recebi aviso de chuva forte e relampagos,chuva_intensa,0.519,orientacao_geral,Acompanhe os avisos da Defesa Civil e da meteo...
3,OC004,Estou preso dentro de casa e preciso de socorro,possivel_urgencia,1.000,encaminhar_imediatamente,"Se houver perigo imediato, afaste-se da area d..."
4,OC005,Como renovar meu documento de identidade?,outros,0.455,fora_escopo,Este prototipo atende apenas orientacoes gerai...
5,OC007,O corrego transbordou e inundou a garagem,alagamento,0.519,orientacao_geral,Evite atravessar vias alagadas e contato com a...
6,OC008,Chuva intensa durante a madrugada,chuva_intensa,0.480,orientacao_geral,Acompanhe os avisos da Defesa Civil e da meteo...
7,OC009,Tem terra deslizando da encosta,deslizamento,0.495,orientacao_geral,"Afaste-se de encostas, barreiras e estruturas ..."
8,OC010,Rua com agua perto do centro. Meu e-mail e [EM...,alagamento,0.388,orientacao_geral,Evite atravessar vias alagadas e contato com a...
9,OC011,Quero uma receita de bolo,outros,0.451,fora_escopo,Este prototipo atende apenas orientacoes gerai...


Resumo de status: {'orientacao_geral': 7, 'encaminhar_imediatamente': 2, 'fora_escopo': 2}


## 6. Resiliência: executar duas vezes e simular uma falha
A repetição na **mesma pasta** deve detectar duplicatas sem inserir linhas novamente. O arquivo ausente deve produzir **código de saída 2**, relatório de erro e log, sem exceção não tratada.

In [5]:
# 6A. Idempotência
r2 = subprocess.run(cmd, cwd=PROJETO, text=True, capture_output=True)
resumo2 = json.loads((DEMO / 'resumo.json').read_text(encoding='utf-8'))
assert r2.returncode == 0
assert resumo2['processados'] == 0
assert resumo2['duplicados'] == RESUMO['processados']
with (DEMO / 'resultados.csv').open(encoding='utf-8-sig', newline='') as f:
    linhas_depois = list(csv.DictReader(f, delimiter=';'))
assert len(linhas_depois) == len(resultados)
print('REEXECUÇÃO SEGURA:', resumo2['duplicados'], 'duplicatas detectadas; nenhuma linha duplicada gravada.')

# 6B. Arquivo ausente
FALHA = SAIDA / 'falha_arquivo_ausente'
cmd_falha = [sys.executable, '-m', 'radar_rpa', 'run', '--input', str(SAIDA / 'nao_existe.csv'), '--output', str(FALHA)]
rf = subprocess.run(cmd_falha, cwd=PROJETO, text=True, capture_output=True)
resumo_falha = json.loads((FALHA / 'resumo.json').read_text(encoding='utf-8'))
assert rf.returncode == 2 and resumo_falha['erros']
assert (FALHA / 'execucao.log').exists()
print('FALHA CONTROLADA: código', rf.returncode, '| relatório:', resumo_falha['erros'])

REEXECUÇÃO SEGURA: 11 duplicatas detectadas; nenhuma linha duplicada gravada.
FALHA CONTROLADA: código 2 | relatório: ['Arquivo de entrada ausente']


## 7. Avaliação da IA com amostra rotulada sintética
O resultado é uma medição real sobre os **16 exemplos sintéticos de teste**, não uma estimativa de desempenho em situações reais.

In [6]:
VALIDACAO = SAIDA / 'validacao'
cmd_val = [sys.executable, '-m', 'radar_rpa', 'validate', '--input', str(PROJETO / 'dados' / 'validacao_rotulada.csv'), '--output', str(VALIDACAO)]
rv = subprocess.run(cmd_val, cwd=PROJETO, text=True, capture_output=True)
print(rv.stdout)
if rv.stderr: print('stderr:', rv.stderr)
assert rv.returncode == 0
medicao = json.loads((VALIDACAO / 'validacao.json').read_text(encoding='utf-8'))
assert medicao['amostra'] == medicao['acertos'] + medicao['erros']
print('Amostra:', medicao['amostra'], '| Acertos:', medicao['acertos'], '| Erros:', medicao['erros'])
print('Limite:', medicao['observacao'])

{
  "amostra": 16,
  "acertos": 16,
  "erros": 0,
  "acuracia": 1.0,
  "observacao": "Conjunto sintetico pequeno. Nao mede desempenho em ocorrencias reais."
}

Amostra: 16 | Acertos: 16 | Erros: 0
Limite: Conjunto sintetico pequeno. Nao mede desempenho em ocorrencias reais.


## 8. Testes automatizados do software
O `pytest` cobre validação de arquivo, anonimização básica, idempotência, urgência, tratamento de falhas e segurança de escrita em CSV.

In [7]:
testes = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], cwd=PROJETO, text=True, capture_output=True)
print(testes.stdout)
if testes.stderr: print(testes.stderr)
assert testes.returncode == 0, 'A suíte automatizada falhou'
print('Testes automatizados aprovados.')

...........                                                              [100%]
11 passed in 1.63s

Testes automatizados aprovados.


## 9. Evidências para entregar
O arquivo ZIP contém `resultados.csv`, `resumo.json`, `execucao.log`, a auditoria SQLite e as evidências dos testes de falha e classificação. Arquivos de saída são gerados com dados **sintéticos** e não precisam ser enviados a nenhum serviço externo.

In [8]:
import shutil
ARQUIVO_ZIP = Path('/content/RADAR_CIDADAO_RPA_EVIDENCIAS_COLAB.zip')
shutil.make_archive(str(ARQUIVO_ZIP.with_suffix('')), 'zip', root_dir=SAIDA)
print('Evidências:', ARQUIVO_ZIP)
print('Commit do código:', COMMIT)
print('Critérios demonstrados: 3+ temas, execução ponta a ponta, falha controlada, logs, validação e testes.')
print('Para baixar, clique no ícone de pasta do Colab e encontre RADAR_CIDADAO_RPA_EVIDENCIAS_COLAB.zip')

Evidências: /content/RADAR_CIDADAO_RPA_EVIDENCIAS_COLAB.zip
Commit do código: 66fa473
Critérios demonstrados: 3+ temas, execução ponta a ponta, falha controlada, logs, validação e testes.
Para baixar, clique no ícone de pasta do Colab e encontre RADAR_CIDADAO_RPA_EVIDENCIAS_COLAB.zip


---
### Ética e limites

Este protótipo utiliza dados fictícios e um classificador local treinado com exemplos sintéticos. A classificação não mede risco real, não é um alerta meteorológico nem substitui a Defesa Civil. O mascaramento de CPF, telefone e e-mail é parcial; não elimina todos os dados identificáveis. O projeto está relacionado ao **ODS 11 — Cidades e Comunidades Sustentáveis** e ao **ODS 13 — Ação Contra a Mudança Global do Clima**.

**Código-fonte e instruções:** [GitHub — rpa-radar-cidadao](https://github.com/sarahmfurlan-sudo/radar-cidadao/tree/main/rpa-radar-cidadao).